In [1]:
import time
import cv2
from movenet_trt import MoveNetTRT, KEYPOINT_NAMES, draw_keypoints
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit  # noqa
import numpy as np

ENGINE_FP16_PATH = "output/pose_fp16.engine"  # main output

# Use a real image if available, else a random image
TEST_IMG = "data/test_imgs/test2.jpg"
if os.path.exists(TEST_IMG):
    frame = cv2.imread(TEST_IMG)
    print("Using {}, shape={}".format(TEST_IMG, frame.shape))
else:
    frame = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
    print("No test.jpg, using random 480x640 image")

# Load engine
movenet_fp16 = MoveNetTRT(ENGINE_FP16_PATH, img_size=192)

Using data/test_imgs/test2.jpg, shape=(408, 612, 3)
[OK] MoveNetTRT loaded: output/pose_fp16.engine
     input : (1, 3, 192, 192)
     output: heatmap (1, 17, 48, 48)
     output: center (1, 1, 48, 48)
     output: regs (1, 34, 48, 48)
     output: offsets (1, 34, 48, 48)


In [2]:
def benchmark(model, frame, n_warmup=10, n_iter=100, label=""):
    # warmup (first calls are slow due to allocation/JIT)
    for _ in range(n_warmup):
        model.infer(frame)

    # timed runs
    times = []
    for _ in range(n_iter):
        t0 = time.time()
        model.infer(frame)
        times.append(time.time() - t0)
    times = np.array(times) * 1000   # to ms

    print("\n=== {} ===".format(label))
    print("  iters: {}".format(n_iter))
    print("  mean : {:.2f} ms  ({:5.1f} FPS)".format(times.mean(), 1000 / times.mean()))
    print("  p50  : {:.2f} ms".format(np.percentile(times, 50)))
    print("  p90  : {:.2f} ms".format(np.percentile(times, 90)))
    print("  p99  : {:.2f} ms".format(np.percentile(times, 99)))
    print("  min  : {:.2f} ms".format(times.min()))
    print("  max  : {:.2f} ms".format(times.max()))
    return times

fp16_times = benchmark(movenet_fp16, frame, label="TRT FP16")


=== TRT FP16 ===
  iters: 100
  mean : 30.59 ms  ( 32.7 FPS)
  p50  : 30.71 ms
  p90  : 33.72 ms
  p99  : 34.91 ms
  min  : 26.53 ms
  max  : 35.11 ms


In [3]:
kpts = movenet_fp16.infer(frame)
print("output shape: {}, dtype: {}".format(kpts.shape, kpts.dtype))
print("\nkeypoints:")
for k, (x, y, s) in enumerate(kpts):
    print("  {:2d} {:14s}  ({:6.1f}, {:6.1f})  score={:.3f}".format(
        k, KEYPOINT_NAMES[k], x, y, s))

# If the input was a real image, save the visualization for visual inspection.
if os.path.exists(TEST_IMG):
    vis = draw_keypoints(frame, kpts)
    cv2.imwrite("trt_test_out.jpg", vis)
    print("\nVisualization saved: trt_test_out.jpg")

output shape: (17, 3), dtype: float32

keypoints:
   0 nose            ( 275.4,   89.8)  score=0.488
   1 left_eye        ( 288.9,   81.8)  score=0.527
   2 right_eye       ( 263.5,   88.9)  score=0.662
   3 left_ear        ( 301.5,   80.9)  score=0.429
   4 right_ear       ( 260.8,   94.4)  score=0.502
   5 left_shoulder   ( 350.3,  106.0)  score=0.401
   6 right_shoulder  ( 273.6,  129.7)  score=0.455
   7 left_elbow      ( 410.8,  121.1)  score=0.231
   8 right_elbow     ( 246.6,  173.7)  score=0.352
   9 left_wrist      ( 422.8,  139.7)  score=0.021
  10 right_wrist     ( 218.9,  189.0)  score=0.195
  11 left_hip        ( 411.5,  190.0)  score=0.494
  12 right_hip       ( 359.9,  206.2)  score=0.446
  13 left_knee       ( 437.5,  248.8)  score=0.248
  14 right_knee      ( 363.0,  265.8)  score=0.307
  15 left_ankle      ( 487.4,  307.9)  score=0.141
  16 right_ankle     ( 411.2,  316.5)  score=0.124

Visualization saved: trt_test_out.jpg


In [6]:
# ============================================================
# SET TO None to skip this comparison.
# Otherwise point at your fire717 .pth file (the one you used
# to generate the ONNX).
# ============================================================
PTH_PATH = "output/movenet.pth"    # e.g. "movenet/output/movenet.pth"

if PTH_PATH is None or not os.path.exists(PTH_PATH or ""):
    print("[SKIP] PyTorch baseline comparison")
    print("       Set PTH_PATH above to enable it.")
else:
    import torch
    # NOTE: this depends on fire717's model class. The cleanest way is to
    # import it from movenet/lib/. Adapt the import to your repo:
    #
    #   sys.path.insert(0, os.path.abspath('movenet'))
    #   from lib.movenet_mobilenetv2 import MoveNet
    #   model = MoveNet(num_classes=17, ft_size=48)
    #   model.load_state_dict(torch.load(PTH_PATH, map_location='cuda'))
    #   model = model.cuda().eval()
    #
    # Once `model` is loaded, re-implement preprocess/postprocess (or
    # reuse fire717's predict.py logic) and benchmark with the same
    # `benchmark()` helper above.
    print("PyTorch baseline scaffolding — fill in the model loading for your repo.")
    print("Expected result on Nano: ~5-10x slower than TRT FP16.")

PyTorch baseline scaffolding — fill in the model loading for your repo.
Expected result on Nano: ~5-10x slower than TRT FP16.
